## **Retrieval Extraction with Embeddings and Cosine Similarity (Without Augmentation & Without Generation)**

Retrieval: Retrieve the most relevant passage or context based on the input question using embeddings and cosine similarity.

Extraction: Extracting the most relevant answer based on embeddings and cosine similarity, without any augmentation or generation steps.

Result: The answer is extracted directly from the context provided by the retrieved passage. The answer is simply the most relevant passage based on similarity.There is no generative model involved.

## Embeddings

**all-MiniLM-L6-v2**: A lightweight, transformer-based model for generating efficient and high-quality sentence embeddings, good for general-purpose NLP tasks.

In [ ]:
!pip install -qqq ollama
!pip install -qqq langchain faiss-cpu python-dotenv
!pip install -qqq -U langchain-openai
!pip install -qqq langchain-core langchain-openai langchain-community
!pip install -qqq matplotlib numpy

In [ ]:
import os
print(os.getcwd())

In [ ]:
import requests
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline, AutoModel, AutoTokenizer
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import ollama
import torch
from langchain_community.embeddings import OllamaEmbeddings
import subprocess
import json
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from transformers import DistilBertForQuestionAnswering, DistilBertTokenizer
from transformers import BartTokenizer, BartForConditionalGeneration
# Suppress warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Ensure directory exists
#os.makedirs('./data/', exist_ok=True)

In [ ]:
# Load the summary from the file
def load_summary(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()

# Main execution
summary_file_path = './msc_ai_hullonline_short.txt'
loaded_summary = load_summary(summary_file_path)

# Print the loaded summary
print(loaded_summary)

## Define Questions

In [ ]:
questions = [
    "What is the mode of delivery for the MSc Artificial Intelligence Online?",
    "How long does the MSc Artificial Intelligence course take to complete?",
    "What is the total cost of the MSc Artificial Intelligence Online program?",
    "When are the start dates for the MSc Artificial Intelligence Online program?",
    "Is there a shorter AI course available besides the MSc Artificial Intelligence Online?",
    "Who is the Programme Director for the MSc in Artificial Intelligence Online at the University of Hull?",
    "How long does the PGCert in Artificial Intelligence take to complete?",
    "What is the total cost of the PGCert in Artificial Intelligence?",
    "What is the minimum academic qualification required to apply for the MSc Artificial Intelligence (Online) program?",
    "What if I don't have a degree but have relevant professional experience?",
    "What language proficiency is required if my first language isn't English?",
    "Who are the lecturers for the MSc Artificial Intelligence program at the University of Hull?",
    "What are the payment options available for tuition at the University of Hull online?",
    "How can I pay my tuition online?",
    "How do I arrange to pay by bank transfer?",
    "What is the WhatsApp number to reach the course advisers at the University of Hull?",
    "When should self-funding students make their tuition payments?",
    "What is the payment deadline for students using student loans?",
    "Where are the students at the University of Hull from?",
    "What kind of community can I expect as an online student at the University of Hull?",
    "What will my student status be once accepted onto an online Masters course at the University of Hull?",
    "Will I receive a University of Hull student ID card as an online student?",
    "What will be on my degree certificate after completing an online Masters course at the University of Hull?",
    "Is there a graduation ceremony for online students at the University of Hull?",
    "What is the phone number to reach the course advisers at the University of Hull?",
    "What is the email address for enquiries about online courses at the University of Hull?",
    "What are the payment options available for paying tuition fees at the University of Hull Online?",
    "Is there any discount available for referring a friend to the University of Hull Online?",
    "What is the total course fee for the MSc in Healthcare Leadership?",
    "How much does the MA in Creative Writing cost?",
    "What is the tuition fee for the PGDip in Healthcare Leadership?",
    "What is the cost of the MSc in Dementia program?",
    "How much does the PGCert in Healthcare Leadership cost?",
    "What is the fee for the PGDip in Dementia?",
    "How much does the MSc in Logistics and Supply Chain Management cost?",
    "What is the tuition fee for the PGCert in Dementia?",
    "What is the total cost of the MSc in People Analytics program?",
    "How much does the MSc in Digital Transformation cost?",
    "What is the fee for the PG Award in People Analytics?",
    "What is the cost of the MA in Education program?",
    "How much does the MSc in Engineering Management cost?",
    "How much does the Global MBA program cost?",
    "What is the University of Hull's ranking in the north of the UK according to the Times Good University Guide 2022?",
    "How did the University of Hull perform in the Guardian University Rankings 2022?",
    "What modules will be covered in the MSc Artificial Intelligence course?",
    "How if I don't have money and have financial problem in the University of Hull's MSc Artificial Intelligence online program? Any loan, scholarship, funding, or financial support?",
    "Are there any exams in the University of Hull's MSc Artificial Intelligence online program?",
    "What happens to my tuition fees if I temporarily suspend my studies at the University of Hull Online?",
    "What happens if I have outstanding financial commitments to Hull Online?",
    "Why study a Master's in Artificial Intelligence at University of Hull online?",
]

ground_truth_answers = [
        "The MSc Artificial Intelligence Online is delivered 100% online.",
        "The MSc Artificial Intelligence Online program takes two years to complete on a part-time basis.",
        "The total cost of the the MSc Artificial Intelligence Online program is £8,950, with instalment and funding options available.",
        "The start dates for the for the MSc Artificial Intelligence Online program are in January, May, and September.",
        "Yes, the 30-week PGCert in Responsible Artificial Intelligence is available for those seeking a shorter course.",
        "The Programme Director for the MSc in Artificial Intelligence Online at the University of Hull is Dr. Rameez Kureshi.",
        "The PGCert Artificial Intelligence program takes 30 weeks to complete on a part-time basis.",
        "The total cost of the PGCert in Artificial Intelligence is £2,750.",
        "The minimum academic qualification required to apply for the MSc Artificial Intelligence Online program is an Honours degree at 2:2 or above (or international equivalent) in a STEM discipline or a closely related subject.",
        "Applicants without relevant degree qualifications will be considered based on relevant professional experience or training, including competence in computer programming.",
        "An IELTS score of 6.0 (with a minimum of 5.5 in each skill) or an equivalent English language proficiency qualification is required.",
        "The lecturers for the MSc Artificial Intelligence program at the University of Hull are Dr. Rameez Kureshi, Professor Adil Khan, Khadija Fraz, Dr. Bhupesh Mishra, Professor Dhaval Thakker",
        "You can pay for tuition online, over the phone using a credit or debit card, or by bank transfer.",
        "To pay online, visit the Convera GlobalPay website after receiving your offer.",
        "To arrange payment by bank transfer, email finance-online@hull.ac.uk after receiving your offer.",
        "You can reach the course advisers at the University of Hull by WhatsApp at  +44 (0)7360 538906",
        "Self-funding students should make the first payment no later than four weeks before the course start date, with subsequent payments due four weeks before each term.",
        "Students using loans must pay their first instalment no later than two weeks after receiving the first loan payment, with subsequent payments due two weeks after each loan payment.",
        "At the University of Hull, students come from over 100 countries around the world, making it a truly global community.",
        "As an online student, you will study alongside peers from across the world and become part of the University's global community.",
        "Once accepted onto an online Masters course, you will be welcomed as a fully-registered student with the University of Hull.",
        "Yes, as a fully-registered student, you will be eligible for a University of Hull student ID card.",
        "Your degree certificate will have the exact same academic weight as that of an on-campus student, with no distinction made between online and on-campus study.",
        "Yes, you will be invited to a graduation ceremony at Hull to celebrate your achievement and receive your degree certificate, identical to that of an on-campus student.",
        "You can reach the course advisers at the University of Hull by phone at +44(0)1482 251 819.",
        "The email address for enquiries about online courses at the University of Hull is enquiries-online@hull.ac.uk.",
        "For the payment options, you can choose to pay fees in full before starting your course or in six convenient instalments over the duration of the course.",
        "Student who refers another student to the University of Hull Online is eligible for a tuition fee discount of up to £750.",
        "The total course fee for the MSc in Healthcare Leadership is £9,950.",
        "The MA in Creative Writing costs £10,600.",
        "The tuition fee for the PGDip in Healthcare Leadership is £6,700.",
        "The cost of the MSc in Dementia program is £10,600.",
        "The PGCert in Healthcare Leadership costs £3,400.",
        "The fee for the PGDip in Dementia is £7,100.",
        "The MSc in Logistics and Supply Chain Management costs £9,950.",
        "The tuition fee for the PGCert in Dementia is £3,600.",
        "The total cost of the MSc in People Analytics program is £10,600.",
        "The MSc in Digital Transformation costs £10,600.",
        "The fee for the PG Award in People Analytics is £1,800.",
        "The cost of the MA in Education program is £8,950.",
        "The MSc in Engineering Management costs £8,950.",
        "The Global MBA program costs £12,150.",
        "The University of Hull is ranked 4th in the north of the UK according to the Times Good University Guide 2022.",
        "The University of Hull climbed 19 places in the Guardian University Rankings 2022.",
        "The MSc Artificial Intelligence course covers the following modules: AI Foundations, Machine Learning & Deep Learning, Ethical Regulatory and Social Aspects for Fair AI, Applied AI, Research/Consultancy Project",
        "If you're experiencing financial or money difficulties, as a University of Hull Online student, you may be eligible for various funding options and bursaries to help cover your course fees. For more information about loans, bursaries, or fee reductions, please call +44(0)1482 251 819 or email enquiries-online@hull.ac.uk.",
        "There are no exams in the University of Hull's MSc Artificial Intelligence online program. All assessments are based on coursework, which is submitted online.",
        "If you temporarily suspend your studies, any tuition fees already paid will be retained until you resume or permanently withdraw. You may be charged again for retaking modules and remain liable for any outstanding fees.",
        "If you have outstanding financial commitments to Hull Online, you won't be able to progress to your next module until your debt is repaid or alternative arrangements are made. To dispute a debt, email finance-online@hull.ac.uk within 7 days of receiving a payment demand.",
        "AI is one of the disruptive innovations of this century that influences all aspects of industries. Our online, part-time MSc in Artificial Intelligence will help you get there. Study from anywhere in the world, at times that suit you best, with expert guidance from our academics and support staff.",
]

## A. Retrieval Extraction with 'all-'MiniLM'-v2' Embeddings and Cosine Similarity:

distilbert-base-cased-distilled-squad model is a question-answering model, not a sentence-transformers model, which is required for generating embeddings.

all-MiniLM-L6-v2' is a lightweight version of a transformer model designed for generating sentence embeddings, based on the architecture of transformer models like BERT, RoBERTa, DistilBERT.

"MiniLM" in the model name refers to Microsoft's MiniLM (Miniature Language Model).

**1. Embedding Process**:
all-MiniLM-L6-v2 used to generate sentence embeddings for both the questions and corresponding answers.
The embeddings created using SentenceTransformer class, and cosine similarity is computed between question embedding and answer embeddings.

**2.Retrieval Process**:

For each question, computes cosine similarity between the question's embedding and each answer's embedding.
The answer with highest cosine similarity score is selected as the most relevant answer.

**3. Evaluation**:

*Exact Match (EM) Score*calculated by comparing each retrieved answer to the ground truth answer. If they match exactly, it's considered a correct match.

*F1 Score*: This score measures the overlap between the tokens in the retrieved answer and the ground truth answer, considering both precision (how much of  retrieved answer is relevant) and recall (how much of relevant answer was retrieved).

**4.Visualization**:

EM and F1 scores are visualized

### A1. Embedding Process

In [ ]:
# Function to generate embeddings, find most relevant answers, save them

def perform_embeddings(embedding_model_name, questions, answers):
    model = SentenceTransformer(embedding_model_name) # Load embedding model
    answer_embeddings = model.encode(answers, convert_to_tensor=True)  # Generate embeddings for answers

    predictions = []

    for question in questions:
        question_embedding = model.encode(question, convert_to_tensor=True) # Generate embedding for the question
        similarities = util.pytorch_cos_sim(question_embedding, answer_embeddings)  # Compute cosine similarity between QnA
        most_similar_index = similarities.argmax() # Find the index of the most similar answer
        most_relevant_answer = answers[most_similar_index]  # Retrieve the most relevant answer

        predictions.append(most_relevant_answer)

        print(f"Q: {question}\nA: {most_relevant_answer}\n")

    return predictions, answer_embeddings

### A2.Retrieval Process:

In [ ]:
# Define model name
embedding_model_name = "all-MiniLM-L6-v2"

# Perform QA with MiniLM embeddings
predictions, MiniLM_embeddings = perform_embeddings(embedding_model_name, questions, ground_truth_answers)

#### Save the "all-MiniLM-L6-v2" embeddings

In [ ]:
# Save embeddings
embedding_file_path = './MiniLM_embeddings.pkl'

with open(embedding_file_path, 'wb') as f:
    pickle.dump(MiniLM_embeddings, f)

print(f"Embeddings saved to {embedding_file_path}")

#### Load and print the saved "all-MiniLM-L6-v2" embeddings from the file

In [ ]:
# Path to embeddings file
embedding_file_path = './MiniLM_embeddings.pkl'

# Load embeddings from file
with open(embedding_file_path, 'rb') as f:
    loaded_embeddings = pickle.load(f)

print(f"Embeddings loaded from {embedding_file_path}:") # Print loaded embeddings
print(loaded_embeddings)

### A3. Evaluation:

### Calculates the Exact Match (EM) Score and F1 Score for a retrieval-based extraction task using all-MiniLM-L6-v2 embeddings and cosine similarity.

Exact Match (EM) Score: Measures proportion of exactly matching answers between  retrievals and the ground truths.

F1 Score: Measures overlap between the retrieved and ground truth answers in terms of shared tokens, balancing precision and recall.

In [ ]:
# Function to calculate Exact Match and F1 Score
def evaluate(predictions, ground_truths):
    em_score = sum([1 if pred == truth else 0 for pred, truth in zip(predictions, ground_truths)]) / len(ground_truths)

    f1_scores = []
    for pred, truth in zip(predictions, ground_truths):
        pred_tokens = pred.split()
        truth_tokens = truth.split()
        common_tokens = set(pred_tokens) & set(truth_tokens)
        if len(common_tokens) == 0:
            f1_scores.append(0)
        else:
            precision = len(common_tokens) / len(pred_tokens)
            recall = len(common_tokens) / len(truth_tokens)
            f1 = 2 * (precision * recall) / (precision + recall)
            f1_scores.append(f1)

    f1_score_avg = sum(f1_scores) / len(f1_scores)

    return em_score, f1_score_avg

# Evaluate the predictions
em_score, f1_score_avg = evaluate(predictions, ground_truth_answers)
print(f"Exact Match (EM) Score: {em_score:.2f}")
print(f"F1 Score: {f1_score_avg:.2f}")

### A4.Visualization:

In [ ]:
# Visualize the EM and F1 scores using a bar chart
scores = [em_score, f1_score_avg]
score_labels = ['Exact Match (EM) Score', 'F1 Score']

plt.figure(figsize=(4, 4))
bars = plt.bar(score_labels, scores, color=['blue', 'green'])
plt.ylim(0, 1)  # Since scores are between 0 and 1
plt.title('Evaluation Scores')
plt.ylabel('Score')

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f'{yval:.2f}', ha='center', va='bottom')

plt.show()

### Interpretation:

The high Exact Match (EM) Score and F1 Score for certain questions whose answers are available in the knowledge base suggests that the retrieval-based model with cosine similarity performs adequately when relevant material is present.



# **Test Questions which were not covered in knowledge base**

In [ ]:
# Define unrelated questions
unrelated_questions = [
    "What is the capital of France?",
    "How does photosynthesis work?",
    "Who invented the light bulb?",
    "What is the square root of 144?",
    "Can you recommend a good recipe for chocolate cake?",
    "What is the current weather in London?",
    "What is the tallest mountain in the world?",
    "How do I change a car tire?"
]

In [ ]:
# Split summary into passages
passages = [line.strip() for line in loaded_summary.split('\n') if line.strip()]
model = SentenceTransformer('all-MiniLM-L6-v2') # Load the SentenceTransformer model

# Generate embeddings for passages
passage_embeddings = model.encode(passages, convert_to_tensor=True)

# Function to retrieve most relevant passage
def retrieve_passage(question, passages, passage_embeddings, model):
    question_embedding = model.encode(question, convert_to_tensor=True)
    cosine_scores = util.pytorch_cos_sim(question_embedding, passage_embeddings)[0]
    best_match_idx = np.argmax(cosine_scores.cpu().numpy())
    return passages[best_match_idx], cosine_scores[best_match_idx].item()

In [ ]:
# Test retrieval performance for unrelated questions
results = []
for q in unrelated_questions:
    retrieved_passage, score = retrieve_passage(q, passages, passage_embeddings, model)
    results.append({
        'question': q,
        'retrieved_passage': retrieved_passage,
        'similarity_score': score
    })

In [ ]:
# Save results to a DataFrame for analysis and visualization
df_results = pd.DataFrame(results)
df_results.to_csv('unrelated_questions_retrieval_results.csv', index=False)

print("Retrieval test completed. Results saved to unrelated_questions_retrieval_results.csv")

In [ ]:
# Visualization
print("\nSimilarity Scores for Unrelated Questions:")
for res in results:
    print(f"Question: {res['question']}\nScore: {res['similarity_score']:.4f}\n")

In [ ]:
plt.figure(figsize=(8, 7))
plt.bar(df_results['question'], df_results['similarity_score'], color='skyblue')
plt.xlabel('Retrieval-Based Methods_unrelated_questions')
plt.ylabel('Score')
plt.title('Retrieval-Based Methods using Embedding and Cosine Similarity on Unrelated Questions')
plt.xticks(rotation=45, fontsize=8)
plt.tight_layout()
plt.savefig('./Retrieval-Based Methods_unrelated_questions.jpeg', format='jpeg', dpi=300)
print("Visualization saved to unrelated_questions_performance.jpeg")

### Interpretation:

For questions not covered in the knowledge base, the Retrieval-Based Methods fails to retrieve or provide accurate answers, highlighting its dependency on existing content and its limitation in handling unseen or out-of-scope queries.